# TreeClass: ID3 and C4.5 Decision Trees

This notebook accompanies the software article. It reproduces the examples of the paper step by step, compares the ID3 and C4.5 implementations with the Weka implementations J48 and Id3, and measures their accuracy and runtime against chefboost and Weka.

## 1. Examples from the Paper

This section reproduces the step-by-step examples detailed in the article on the Play Tennis dataset. The script prints every intermediate value reported in the paper and then checks all of them automatically:
* **ID3 (categorical data):** entropy of the full set, information gain of every attribute, partitioning by *Outlook* and the recursive splits of the *Sunny* and *Rain* subsets
* **C4.5 (numeric Temperature and Humidity):** candidate thresholds at midpoints, the table of gain, split information and gain ratio at every node, the average-gain safeguard and the selected test
* **Resulting trees:** the induced ID3 tree and the C4.5 tree before and after error-based pruning (CF = 25%)

In [ ]:
# Execute the verbose test script to trace every intermediate step of both examples
!python tests/test_article_examples.py

## 2. Datasets

Besides Play Tennis, three public UCI datasets are used. Each one covers a different aspect of the algorithms:
* **Iris** - numeric attributes only, which exercises threshold selection in C4.5
* **Mushroom** - categorical attributes, the largest dataset (8,124 cases), with missing values in *stalk-root*
* **Congressional Voting Records** - categorical attributes with many missing values and noisy classes, which exercises the fractional weights of missing values and error-based pruning

In [ ]:
import sys
import pandas as pd
from IPython.display import display

sys.path[:0] = ["src", "benchmarks"]
import benchmark as bm

datasets = {
    "Play Tennis": (pd.read_csv("data/play_tennis_numeric.csv").drop(columns="Day"), "Play"),
    "Iris": (bm.load_iris(), bm.TARGET),
    "Mushroom": (bm.load_mushroom(), bm.TARGET),
    "Votes": (bm.load_votes(), bm.TARGET),
}

summary = []
for name, (data, target) in datasets.items():
    features = data.drop(columns=target)
    summary.append({
        "Dataset": name,
        "Cases": len(data),
        "Attributes": features.shape[1],
        "Numeric": sum(pd.api.types.is_numeric_dtype(features[c]) for c in features),
        "Classes": data[target].nunique(),
        "Missing Values": int(features.isna().sum().sum()),
    })

display(pd.DataFrame(summary))

## 3. Functional Verification against Reference Implementations

To verify the correctness of the implementations, cross-validation is performed against Weka: TreeClass C4.5 is compared with **J48** (configured as close as possible: CF = 25%, minimum 1 case per branch, no subtree raising, no MDL correction, thresholds at midpoints) and TreeClass ID3 with **Id3**. Both tools receive identical stratified folds. Beyond accuracy, the trees built on the full data are compared line by line.

Expected outcome: identical trees on the categorical datasets (Mushroom, Votes). On Iris, the trees may differ, because J48 implements C4.5 Release 8, which selects numeric thresholds by gain instead of gain ratio.

In [ ]:
print("Executing TreeClass and Weka on identical folds... (please wait)", flush=True)
# 1. Run the validation suite (Java is required; Weka jars are fetched on first use)
!python -u benchmarks/j48_validation.py --folds 10

print("\nLoading and cross-verifying the induced trees...", flush=True)
# 2. Load and summarise the comparison
results = pd.read_csv("output/j48_validation.csv")
display(results.round(4))

print("\n[VERIFICATION SUMMARY]")
for _, row in results.iterrows():
    status = "IDENTICAL TREES" if row["same_tree"] else "TREES DIFFER"
    print(f"{row['dataset']:<9} {row['variant']:<22} agreement {row['agreement']:.4f} -> {status}")

categorical = results[results["dataset"] != "iris"]
if categorical["same_tree"].all():
    print("\nSUCCESS: On categorical data TreeClass builds exactly the same trees as Weka J48 and Id3.")
else:
    print("\nERROR: Mismatch detected in the trees built on categorical data.")

The trees built on the full data are saved in `output/` (e.g. `iris_pruned_tree_j48.txt` and `iris_pruned_tree_treeclass.txt`), so they can be read side by side. The cell below shows the Iris trees, where the two releases of C4.5 choose different thresholds.

In [ ]:
for tool in ("treeclass", "j48"):
    print(f"=== Iris, pruned C4.5 tree: {tool} ===")
    print(open(f"output/iris_pruned_tree_{tool}.txt").read())

## 4. Performance Benchmarking

The predictive quality and runtime are evaluated by executing the benchmark suite on all three UCI datasets with 10-fold stratified cross-validation. Every tool receives the same folds:
1. **TreeClass:** ID3, C4.5 before pruning and C4.5 after pruning
2. **chefboost:** ID3 and C4.5 (a Python decision tree library)
3. **Weka:** Id3 and J48 before and after pruning (run through Java; the build time is the one reported by Weka, so the start of the Java virtual machine is not counted)

In [ ]:
print("Benchmarking all implementations... This might take several minutes.", flush=True)
!python -u benchmarks/benchmark.py --folds 10 --out output/benchmark.csv
print("\nBenchmark suite finished!", flush=True)

## 5. Performance Analysis and Metrics

This section loads the benchmark results. The data is presented in a tabular format for exact metric inspection (accuracy, macro-F1, tree size, build and prediction time) and plotted to compare the build time of the tools on each dataset.

In [ ]:
import matplotlib.pyplot as plt

csv_path = "output/benchmark.csv"

try:
    df = pd.read_csv(csv_path).dropna(subset=["accuracy"])

    display_df = df[["dataset", "method", "accuracy", "macro_f1", "nodes", "leaves", "depth",
                     "fit_ms", "predict_ms"]].copy()
    display_df.columns = ["Dataset", "Method", "Accuracy", "Macro-F1", "Nodes", "Leaves",
                          "Depth", "Build Time (ms)", "Prediction Time (ms)"]

    print("=== BENCHMARK METRICS (10-fold cross-validation, mean over folds) ===")
    display(display_df)

    # Build time of every method on every dataset (log scale, as Weka runs in Java)
    times = df.pivot_table(index="method", columns="dataset", values="fit_ms")
    ax = times.plot.bar(figsize=(12, 6), logy=True, rot=30)
    ax.set_title("Build Time per Dataset")
    ax.set_xlabel("Method")
    ax.set_ylabel("Build Time (ms, log scale)")
    ax.legend(title="Dataset")
    ax.grid(True, axis="y", linestyle="--", alpha=0.6)
    plt.tight_layout()
    plt.show()

except FileNotFoundError:
    print(f"Error: Benchmark results file not found at {csv_path}. Please execute the benchmark suite first.")

## 6. Depth Limit of chefboost

chefboost stops splitting at depth 5 by default (`max_depth = 5`), also for ID3 and C4.5, and creates a leaf with the majority class there. This explains why its ID3 differs from TreeClass and Weka Id3 on Votes, where the ID3 trees have depth 7.6 on average, but not on Mushroom, where they have depth 4.

The cell below runs chefboost ID3 on Votes with its default settings and without the limit (`max_depth = 1000`), next to TreeClass ID3, on the same folds as the benchmark. Expected outcome: without the limit, chefboost has the same tree depth as TreeClass, and its predictions differ from TreeClass on one of the 435 test cases. The script `benchmarks/check_chefboost_depth.py` runs the same check for all datasets and both algorithms.

In [ ]:
import check_chefboost_depth as cd

depth_check = pd.DataFrame(cd.check("votes", "ID3"))
display(depth_check)

The remaining test case is traced below through the TreeClass tree. It reaches a node with one training case of each class, where two attributes have the same gain, so the prediction depends on how ties are resolved. TreeClass selects the attribute that comes first in the data and, for equal class counts, the class that comes first in alphabetical order; chefboost resolves ties differently.

In [ ]:
!python -u benchmarks/check_chefboost_case.py

## 7. Scalability Analysis

This final section evaluates how predictive quality and build time change with the number of cases. The benchmark is executed on stratified random samples of the Mushroom dataset (10%, 25%, 50% and 100% of the cases); every tool receives the same samples and the same folds.

In [ ]:
print("Benchmarking on growing samples of Mushroom... This might take several minutes.", flush=True)
!python -u benchmarks/benchmark.py --datasets mushroom --fractions 0.1 0.25 0.5 1 --folds 10 --out output/benchmark_scaling.csv
print("\nScalability suite finished!", flush=True)

In [ ]:
csv_path = "output/benchmark_scaling.csv"

try:
    df = pd.read_csv(csv_path).dropna(subset=["accuracy"])

    display_df = df[["cases", "method", "accuracy", "nodes", "fit_ms", "predict_ms"]].copy()
    display_df.columns = ["Cases", "Method", "Accuracy", "Nodes", "Build Time (ms)", "Prediction Time (ms)"]
    print("=== SCALABILITY METRICS (Mushroom) ===")
    display(display_df)

    plt.figure(figsize=(10, 6))
    for method in df["method"].unique():
        subset = df[df["method"] == method].sort_values("cases")
        plt.plot(subset["cases"], subset["fit_ms"], marker="o", linewidth=2, label=method)

    plt.yscale("log")
    plt.title("Build Time vs. Number of Cases (Mushroom)")
    plt.xlabel("Number of Cases")
    plt.ylabel("Build Time (ms, log scale)")
    plt.legend(title="Method", bbox_to_anchor=(1.05, 1), loc="upper left")
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.tight_layout()
    plt.show()

except FileNotFoundError:
    print(f"Error: Scalability results not found at {csv_path}. Please execute Section 7 first.")